# Notebook 05 — Fraud Detection Platform: Deepened Stress-Testing & Scenario Analysis
**Master Playbook Section 7/19, gap-analysis priority 5 — deepens Notebook 02's single hypothetical scenario call into a named 3-tier ladder, a fully vectorized multi-scenario grid (105 combinations, no Python loop), a real reverse stress test solving for the fraud-rate multiplier that breaches a stated loss tolerance, and a genuine re-scoring of the real trained champion model under a disclosed Amount shock. No retraining. Every multiplier is a labeled ASSUMPTION or a real, sourced constant already used elsewhere in this project — nothing invented silently.**


In [ ]:
# ============================================================
# SETUP -- WARP-optimized environment (thread ceiling set BEFORE any ML import)
# CPU/RAM thresholds: CPU 93% (90-95% band), RAM 90%.
# ============================================================
import os, time, json, pickle, warnings, subprocess, sys
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

_RUN_T0 = time.time()

CPU_THRESHOLD_PCT = 93
RAM_THRESHOLD_PCT = 90

_N_THREADS = max(1, int((os.cpu_count() or 4) * (CPU_THRESHOLD_PCT / 100) // 1))
os.environ.setdefault("OMP_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("POLARS_MAX_THREADS", str(_N_THREADS))

# Lean auto-install guard -- this notebook needs nothing beyond what NB1-04
# already required: no fastapi/httpx (no API calls here), no shap/xgboost/
# lightgbm (no training, no explainability). Vectorized arithmetic + one
# real model's predict_proba only.
for _pkg in ("polars", "psutil", "pyarrow", "catboost"):
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", _pkg], check=True)

import polars as pl
import psutil
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

_ram_start = psutil.virtual_memory()
print(f"WARP thread ceiling: {_N_THREADS} threads (of {os.cpu_count()} available cores, target {CPU_THRESHOLD_PCT}%)")
print(f"RAM at startup: {_ram_start.percent:.1f}% used ({_ram_start.used/1e9:.2f} GB / {_ram_start.total/1e9:.2f} GB) -- target ceiling {RAM_THRESHOLD_PCT}%")
if _ram_start.percent >= RAM_THRESHOLD_PCT:
    print(f"WARNING: RAM already at/above the {RAM_THRESHOLD_PCT}% target before this notebook has loaded any data.")
print("Setup complete.")

##############################################################################
# REPO-LAYOUT BOOTSTRAP -- verified detection, unchanged from NB1-04.
##############################################################################
_KNOWN_REPO_ROOT = r"C:\Users\rnand\Downloads\Fraud_Detection_Platform\Fraud_Detection_Platform_repo_only"

def _looks_like_repo(_p):
    return os.path.isdir(os.path.join(_p, "notebooks")) or os.path.exists(os.path.join(_p, "requirements.txt"))

_cwd = os.getcwd()
_parent = os.path.abspath(os.path.join(_cwd, ".."))

if os.path.isdir(_KNOWN_REPO_ROOT):
    REPO_ROOT = _KNOWN_REPO_ROOT
elif _looks_like_repo(_parent):
    REPO_ROOT = _parent
elif _looks_like_repo(_cwd):
    REPO_ROOT = _cwd
else:
    REPO_ROOT = _cwd

_SCAFFOLD_DIRS = [
    "data/raw", "data/processed", "notebooks/starters",
    "src", "deployment", "reports", "docs", "publish_drafts", "tests",
]
for _rel in _SCAFFOLD_DIRS:
    try:
        os.makedirs(os.path.join(REPO_ROOT, *_rel.split("/")), exist_ok=True)
    except PermissionError as _e:
        print(f"WARNING: could not create '{_rel}' under {REPO_ROOT} ({_e}). Skipping.")

NB1_RESULTS_DIR = os.path.join(REPO_ROOT, "reports", "nb1_results")
RESULTS_DIR = os.path.join(REPO_ROOT, "reports", "nb5_results")
try:
    os.makedirs(RESULTS_DIR, exist_ok=True)
except PermissionError:
    RESULTS_DIR = os.path.join(_cwd, "nb5_results")
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f"WARNING: falling back to {RESULTS_DIR} (no write access to {REPO_ROOT}).")

print(f"Repo root:      {REPO_ROOT}")
print(f"NB1 results in: {NB1_RESULTS_DIR}")
print(f"NB5 results in: {RESULTS_DIR}")

##############################################################################
# SELF-CONTAINED MODULE BOOTSTRAP -- two_gate_validation.py only (its
# stress_test_scenario function is reused verbatim for the named-tier ladder
# in Section A, so Section B's vectorized grid can be checked against it).
##############################################################################
from pathlib import Path as _Path

_HERE = _Path.cwd()
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

_MODULE_SOURCES = {
    "two_gate_validation.py": '"""\ntwo_gate_validation.py\nReusable module — implements Section 7 of the Master Playbook: the two-gate\nvalidation architecture, extended with the Stress-Test Scenario sub-gate.\n\nGate 1 = structural integrity (always computable; passing does not mean the\nmodel is good, only that the pipeline ran correctly).\nGate 2 = statistical robustness + concentration + stress-test scenario\n(the gate that can actually fail and should block promotion).\n\nThis module reports REAL numbers only — it never asserts a verdict for you;\nit returns the evidence and a pass/fail against thresholds YOU set explicitly,\nso no threshold is silently assumed.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom dataclasses import dataclass, field\n\n\n# ---------------------------------------------------------------------------\n# GATE 1 — Structural Integrity\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass Gate1Result:\n    checks: dict[str, bool] = field(default_factory=dict)\n\n    @property\n    def all_passed(self) -> bool:\n        return all(self.checks.values())\n\n\ndef run_gate1_structural_checks(df: pd.DataFrame, required_columns: list[str],\n                                 label_column: str) -> Gate1Result:\n    result = Gate1Result()\n\n    result.checks["schema_has_required_columns"] = all(\n        c in df.columns for c in required_columns\n    )\n    result.checks["no_null_in_label"] = df[label_column].isna().sum() == 0\n    result.checks["label_is_binary"] = set(df[label_column].unique()).issubset({0, 1})\n    result.checks["row_count_positive"] = len(df) > 0\n    # No-leakage sanity check: label must not equal a trivial function of\n    # itself once cast (catches an accidental duplicate/label-copy column).\n    numeric_cols = df.select_dtypes(include=[np.number]).columns\n    suspicious_leak_cols = [\n        c for c in numeric_cols\n        if c != label_column and df[c].corr(df[label_column]) > 0.999\n    ]\n    result.checks["no_perfect_correlation_leakage"] = len(suspicious_leak_cols) == 0\n    if suspicious_leak_cols:\n        result.checks["_leak_columns_found"] = suspicious_leak_cols  # diagnostic, not boolean\n\n    return result\n\n\n# ---------------------------------------------------------------------------\n# GATE 2 — Statistical Robustness / Concentration / Stress-Test\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass Gate2Result:\n    cv_auc_scores: list[float]\n    cv_pr_auc_scores: list[float]\n    concentration_report: pd.DataFrame\n    stress_test_report: dict\n\n    @property\n    def cv_stability_ok(self, max_std: float = 0.03) -> bool:\n        return float(np.std(self.cv_pr_auc_scores)) <= max_std\n\n\ndef concentration_report_by_amount_and_time(\n    df: pd.DataFrame, amount_col: str, time_col: str,\n    y_true_col: str, y_pred_col: str, n_amount_bands: int = 5,\n) -> pd.DataFrame:\n    """\n    Real segment-cut breakdown replacing the demographic cuts unavailable on\n    this anonymized dataset (Section 2\'s disclosed limitation) — false-positive\n    and false-negative rates broken out by Amount band and hour-of-day, so\n    concentration risk is checked on the real segments that ARE available.\n    """\n    work = df.copy()\n    work["_amount_band"] = pd.qcut(work[amount_col], n_amount_bands, duplicates="drop")\n    work["_hour_of_day"] = (work[time_col] // 3600) % 24\n\n    rows = []\n    for (band, hour), g in work.groupby(["_amount_band", "_hour_of_day"], observed=True):\n        fp = ((g[y_true_col] == 0) & (g[y_pred_col] == 1)).sum()\n        fn = ((g[y_true_col] == 1) & (g[y_pred_col] == 0)).sum()\n        n = len(g)\n        rows.append({\n            "amount_band": str(band), "hour_of_day": int(hour), "n": n,\n            "false_positive_rate": fp / n if n else np.nan,\n            "false_negative_rate": fn / n if n else np.nan,\n        })\n    return pd.DataFrame(rows)\n\n\ndef stress_test_scenario(\n    df: pd.DataFrame, amount_col: str, scenario_volume_multiplier: float,\n    scenario_fraud_rate_multiplier: float, base_fraud_rate: float,\n) -> dict:\n    """\n    Real, disclosed hypothetical scenario per Section 7/19: since the exact\n    confidential Fed severely-adverse scenario file is not available to a\n    portfolio project, this applies a documented, labeled hypothetical shock\n    (e.g., 2x transaction volume, 3x fraud rate during a stress event) and\n    projects the resulting loss impact on THIS dataset\'s real Amount\n    distribution. Every multiplier is a stated ASSUMPTION, never invented\n    silently — pass real historical-analogue multipliers if you have them.\n    """\n    projected_fraud_rate = base_fraud_rate * scenario_fraud_rate_multiplier\n    projected_transaction_count = len(df) * scenario_volume_multiplier\n    avg_amount = df[amount_col].mean()\n\n    projected_fraud_count = projected_transaction_count * projected_fraud_rate\n    projected_loss = projected_fraud_count * avg_amount\n\n    return {\n        "ASSUMPTION_scenario_volume_multiplier": scenario_volume_multiplier,\n        "ASSUMPTION_scenario_fraud_rate_multiplier": scenario_fraud_rate_multiplier,\n        "base_fraud_rate_real": base_fraud_rate,\n        "projected_fraud_rate_under_scenario": projected_fraud_rate,\n        "projected_transaction_count": projected_transaction_count,\n        "projected_fraud_loss_usd_or_eur": float(projected_loss),\n    }\n',
}
for _fname, _src in _MODULE_SOURCES.items():
    (_HERE / _fname).write_text(_src, encoding="utf-8")
for _modname in ("two_gate_validation",):
    sys.modules.pop(_modname, None)

import two_gate_validation as tgv

print("Self-installed local modules (UTF-8):", ", ".join(_MODULE_SOURCES))

##############################################################################
# LOAD NOTEBOOK 01's REAL OUTPUTS -- no retraining.
##############################################################################
with open(os.path.join(NB1_RESULTS_DIR, "nb1_final_results.json"), encoding="utf-8") as f:
    nb1_results = json.load(f)
with open(os.path.join(NB1_RESULTS_DIR, "champion_model.pkl"), "rb") as f:
    champion_model = pickle.load(f)

FEATURE_COLS = list(champion_model.feature_names_)
CHOSEN_THRESHOLD = nb1_results["threshold_result"]["threshold"]
BASE_FRAUD_RATE = nb1_results["dataset"]["fraud_rate"]
REAL_CURRENT_TOTAL_COST_EUR = nb1_results["threshold_result"]["total_cost"]
print(f"Loaded champion model ({nb1_results['champion_name']}), operating threshold {CHOSEN_THRESHOLD:.4f}")
print(f"Real current base fraud rate: {BASE_FRAUD_RATE:.6%}  |  real current total cost at this threshold: EUR {REAL_CURRENT_TOTAL_COST_EUR:,.2f}")

##############################################################################
# DATA_PATH resolution + Polars-accelerated load -- NB1-04's fixes reused
# unchanged.
##############################################################################
DATA_PATH = None
for _cand in [
    r"C:\Users\rnand\Downloads\creditcard.csv\creditcard.csv",  # your real dataset location -- checked first
    "creditcard.csv",
    os.path.join("data", "raw", "creditcard.csv"),
    os.path.join("..", "data", "raw", "creditcard.csv"),
    os.path.join("..", "creditcard.csv"),
]:
    if os.path.exists(_cand):
        DATA_PATH = _cand
        break
if DATA_PATH is None:
    raise FileNotFoundError(
        "creditcard.csv not found. Place it next to this notebook, or at "
        "data/raw/creditcard.csv relative to the repo root."
    )
print("Using DATA_PATH:", DATA_PATH)

_SCHEMA_OVERRIDES = {"Time": pl.Float64, "Amount": pl.Float64, "Class": pl.Int64}
for _i in range(1, 29):
    _SCHEMA_OVERRIDES[f"V{_i}"] = pl.Float64

_t_load0 = time.time()
df = pl.read_csv(DATA_PATH, schema_overrides=_SCHEMA_OVERRIDES).to_pandas()
print(f"Loaded {len(df):,} rows x {len(df.columns)} cols via Polars in {time.time()-_t_load0:.3f}s")
_ram_after_load = psutil.virtual_memory()
print(f"RAM after load: {_ram_after_load.percent:.1f}% used ({_ram_after_load.used/1e9:.2f} GB / {_ram_after_load.total/1e9:.2f} GB)")

N_ROWS = len(df)
AVG_AMOUNT = float(df["Amount"].mean())
EUR_TO_USD = 1.1592  # real ECB euro foreign exchange reference rate, 2026-09-11 fixing -- same constant used project-wide

# Real, sourced cost multipliers (Master Playbook Section 10, already applied
# in NB1's real cost-optimal threshold search -- reused verbatim here, not
# re-derived or re-guessed):
#   FN_COST_PER_DOLLAR_LOST = 4.41  -- LexisNexis "True Cost of Fraud" multiplier
#   FP_COST_MULTIPLIER_PCT  = 9.2   -- relative severity of false declines vs
#                                       fraud losses, industry-wide (Aite-Novarica /
#                                       Statista via Riskified), applied as an
#                                       operational-cost percentage of flagged amount
FN_COST_PER_DOLLAR_LOST = 4.41
FP_COST_MULTIPLIER_PCT = 9.2

print("=" * 70)
print("SECTION A -- NAMED SCENARIO LADDER (real module, tgv.stress_test_scenario)")
print("=" * 70)

_TIERS = [
    ("baseline", 1.0, 1.0,
     "Today's real observed volume and fraud rate -- no shock applied."),
    ("moderate", 2.0, 3.0,
     "Same disclosed ASSUMPTION multipliers NB2 used, kept unchanged here for direct comparability across notebooks."),
    ("severe", 3.0, 6.0,
     "New, deeper ASSUMPTION tier added in NB5. Loosely informed by -- NOT literally derived from -- the severity "
     "ranking of the Federal Reserve's 2026 CCAR severely-adverse macro scenario already cited in this project's own "
     "gap-analysis doc (unemployment +5.5pp to 10% by Q3 2027, BBB corporate spread +4.4pp, CRE -40%). No published, "
     "verified model exists mapping those specific macro variables to a card-fraud volume/rate multiplier at this "
     "portfolio-project's scope, so this stays a clearly labeled ASSUMPTION, never presented as a citation-backed figure."),
]

scenario_ladder = []
for _name, _vol, _fr, _note in _TIERS:
    _r = tgv.stress_test_scenario(df, "Amount", _vol, _fr, BASE_FRAUD_RATE)
    _r["tier_name"] = _name
    _r["tier_note"] = _note
    _r["projected_fraud_loss_usd"] = _r["projected_fraud_loss_usd_or_eur"] * EUR_TO_USD
    scenario_ladder.append(_r)
    print(f"  [{_name:>8s}] vol x{_vol:.1f} fraud-rate x{_fr:.1f}  ->  "
          f"EUR {_r['projected_fraud_loss_usd_or_eur']:,.2f}  |  USD {_r['projected_fraud_loss_usd']:,.2f}")

print("=" * 70)
print("SECTION B -- VECTORIZED MULTI-SCENARIO GRID (no Python loop over combinations)")
print("=" * 70)

_t_grid0 = time.time()
volume_grid = np.arange(1.0, 4.01, 0.5)   # 7 values: 1.0x .. 4.0x
fraud_grid = np.arange(1.0, 8.01, 0.5)    # 15 values: 1.0x .. 8.0x
VOL, FR = np.meshgrid(volume_grid, fraud_grid, indexing="ij")  # shape (7, 15)

projected_txn_count_grid = N_ROWS * VOL
projected_fraud_rate_grid = BASE_FRAUD_RATE * FR
projected_loss_eur_grid = projected_txn_count_grid * projected_fraud_rate_grid * AVG_AMOUNT
projected_loss_usd_grid = projected_loss_eur_grid * EUR_TO_USD
_grid_seconds = time.time() - _t_grid0

# Sanity cross-check: the grid cell at vol=2.0x/fraud=3.0x must match the real
# module's own "moderate" tier value -- proves the vectorized formula is the
# exact same real arithmetic, not a re-derived approximation.
_i_check = np.where(np.isclose(volume_grid, 2.0))[0][0]
_j_check = np.where(np.isclose(fraud_grid, 3.0))[0][0]
_grid_moderate_eur = float(projected_loss_eur_grid[_i_check, _j_check])
_module_moderate_eur = scenario_ladder[1]["projected_fraud_loss_usd_or_eur"]
_grid_matches_module = bool(np.isclose(_grid_moderate_eur, _module_moderate_eur, rtol=1e-9))
print(f"  Grid shape: {VOL.shape} ({VOL.size} scenario combinations), computed in {_grid_seconds*1000:.2f} ms")
print(f"  Cross-check vs. real module (vol=2.0x, fraud=3.0x): grid={_grid_moderate_eur:,.2f}  "
      f"module={_module_moderate_eur:,.2f}  MATCH: {_grid_matches_module}")

_worst_idx = np.unravel_index(np.argmax(projected_loss_eur_grid), projected_loss_eur_grid.shape)
_worst_vol, _worst_fr = float(VOL[_worst_idx]), float(FR[_worst_idx])
_worst_loss_eur = float(projected_loss_eur_grid[_worst_idx])
_worst_loss_usd = float(projected_loss_usd_grid[_worst_idx])
print(f"  Worst grid combination tested: vol x{_worst_vol:.1f}, fraud-rate x{_worst_fr:.1f}  ->  "
      f"EUR {_worst_loss_eur:,.2f}  |  USD {_worst_loss_usd:,.2f}")

print("=" * 70)
print("SECTION C -- REVERSE STRESS TEST (real vectorized search, volume held at 1.0x)")
print("=" * 70)

_t_reverse0 = time.time()
fraud_mult_fine = np.arange(1.0, 30.0 + 1e-9, 0.01)  # 2,901 values -- still vectorized/instant
projected_loss_fine_eur = N_ROWS * 1.0 * (BASE_FRAUD_RATE * fraud_mult_fine) * AVG_AMOUNT
_reverse_seconds = time.time() - _t_reverse0

_breach_targets = [2, 3, 5, 10]
reverse_stress_results = []
for _mult in _breach_targets:
    _target_eur = REAL_CURRENT_TOTAL_COST_EUR * _mult
    _breach_mask = projected_loss_fine_eur >= _target_eur
    if _breach_mask.any():
        _breach_fr = float(fraud_mult_fine[np.argmax(_breach_mask)])
        _breach_rate = BASE_FRAUD_RATE * _breach_fr
        reverse_stress_results.append({
            "target_multiple_of_current_cost": _mult, "target_cost_eur": _target_eur,
            "required_fraud_rate_multiplier": _breach_fr, "required_fraud_rate": _breach_rate,
            "reached_within_tested_range": True,
        })
        print(f"  {_mult}x today's real total cost (EUR {_target_eur:,.2f}) is breached at "
              f"fraud-rate x{_breach_fr:.2f} (fraud rate rising from {BASE_FRAUD_RATE:.4%} to {_breach_rate:.4%})")
    else:
        reverse_stress_results.append({
            "target_multiple_of_current_cost": _mult, "target_cost_eur": _target_eur,
            "required_fraud_rate_multiplier": None, "required_fraud_rate": None,
            "reached_within_tested_range": False,
        })
        print(f"  {_mult}x today's real total cost NOT reached within the tested range (up to fraud-rate x30.0)")

print("=" * 70)
print("SECTION D -- MODEL-BASED RE-SCORING UNDER A REAL AMOUNT-SHOCK (genuinely re-runs the real model)")
print("=" * 70)
print("  Methodology caveat (same disclosed limitation as NB2's adversarial-robustness section): scored in-sample "
      "(the real champion model scored on rows it was fit on), not a strictly held-out fold -- the shock=0% row below "
      "will NOT exactly match NB1's real out-of-fold confusion matrix. Treat this section as a directional sensitivity "
      "signal, not a replacement for NB1's real out-of-fold headline metrics.")

_t_rescroing0 = time.time()
_y_true = df["Class"].to_numpy()
_shock_levels = [0.0, 0.10, 0.25, 0.50]  # ASSUMPTION: hypothetical average-transaction-value inflation under stress -- not tied to a specific external statistic
model_rescoring_results = []
for _shock in _shock_levels:
    _X = df[FEATURE_COLS].copy()
    _shocked_amount = (df["Amount"].to_numpy() * (1.0 + _shock)).clip(min=0.0)
    _X["Amount"] = _shocked_amount
    _scores = champion_model.predict_proba(_X)[:, 1]
    _preds = (_scores >= CHOSEN_THRESHOLD).astype(int)

    _tp_mask = (_y_true == 1) & (_preds == 1)
    _fp_mask = (_y_true == 0) & (_preds == 1)
    _fn_mask = (_y_true == 1) & (_preds == 0)
    _tn_mask = (_y_true == 0) & (_preds == 0)
    _tp, _fp, _fn, _tn = int(_tp_mask.sum()), int(_fp_mask.sum()), int(_fn_mask.sum()), int(_tn_mask.sum())
    _precision = _tp / (_tp + _fp) if (_tp + _fp) > 0 else float("nan")
    _recall = _tp / (_tp + _fn) if (_tp + _fn) > 0 else float("nan")

    _fn_amt = float(_shocked_amount[_fn_mask].sum())
    _fp_amt = float(_shocked_amount[_fp_mask].sum())
    _fn_cost = _fn_amt * FN_COST_PER_DOLLAR_LOST
    _fp_cost = _fp_amt * FP_COST_MULTIPLIER_PCT / 100.0
    _total_cost_eur = _fn_cost + _fp_cost

    model_rescoring_results.append({
        "amount_shock_pct": _shock, "tp": _tp, "fp": _fp, "fn": _fn, "tn": _tn,
        "precision": _precision, "recall": _recall,
        "fn_cost_eur": _fn_cost, "fp_cost_eur": _fp_cost, "total_cost_eur": _total_cost_eur,
        "total_cost_usd": _total_cost_eur * EUR_TO_USD,
    })
    print(f"  shock +{_shock:.0%}: TP={_tp} FP={_fp} FN={_fn}  precision={_precision:.4f}  recall={_recall:.4f}  "
          f"total_cost=EUR {_total_cost_eur:,.2f} (USD {_total_cost_eur*EUR_TO_USD:,.2f})")

_rescoring_seconds = time.time() - _t_rescroing0
_baseline_cost = model_rescoring_results[0]["total_cost_eur"]
for _r in model_rescoring_results:
    _r["delta_cost_vs_shock0_eur"] = _r["total_cost_eur"] - _baseline_cost
print(f"  4 real vectorized predict_proba calls over {N_ROWS:,} rows each, completed in {_rescoring_seconds:.3f}s total.")

##############################################################################
# SAVE NOTEBOOK 05 RESULTS
##############################################################################
nb5_report = {
    "run_metadata": {"random_seed": RANDOM_SEED, "n_threads": _N_THREADS,
                     "chosen_threshold": CHOSEN_THRESHOLD, "base_fraud_rate": BASE_FRAUD_RATE,
                     "eur_to_usd_rate": EUR_TO_USD, "eur_to_usd_source": "ECB euro foreign exchange reference rate, 2026-09-11 fixing",
                     "fn_cost_per_dollar_lost": FN_COST_PER_DOLLAR_LOST, "fn_cost_source": "LexisNexis True Cost of Fraud multiplier",
                     "fp_cost_multiplier_pct": FP_COST_MULTIPLIER_PCT, "fp_cost_source": "Aite-Novarica/Statista via Riskified"},
    "scenario_ladder": scenario_ladder,
    "grid_sweep": {
        "volume_multipliers": volume_grid.tolist(), "fraud_rate_multipliers": fraud_grid.tolist(),
        "n_combinations": int(VOL.size), "grid_compute_seconds": round(_grid_seconds, 6),
        "cross_check_vs_module_passed": _grid_matches_module,
        "worst_combination": {"volume_multiplier": _worst_vol, "fraud_rate_multiplier": _worst_fr,
                              "projected_loss_eur": _worst_loss_eur, "projected_loss_usd": _worst_loss_usd},
    },
    "reverse_stress_test": {
        "volume_held_at": 1.0, "search_range_fraud_multiplier": [1.0, 30.0], "search_step": 0.01,
        "search_compute_seconds": round(_reverse_seconds, 6),
        "results": reverse_stress_results,
    },
    "model_based_rescoring_under_amount_shock": {
        "caveat": "In-sample scoring (same disclosed limitation as NB2's adversarial-robustness section) -- directional sensitivity signal, not a replacement for NB1's real out-of-fold headline metrics.",
        "compute_seconds": round(_rescoring_seconds, 4),
        "results": model_rescoring_results,
    },
}
with open(os.path.join(RESULTS_DIR, "nb5_stress_test_report.json"), "w", encoding="utf-8") as f:
    json.dump(nb5_report, f, indent=2, default=str)

_ram_end = psutil.virtual_memory()
_total_elapsed = time.time() - _RUN_T0
print("=" * 70)
print(f"RAM at finish: {_ram_end.percent:.1f}% used ({_ram_end.used/1e9:.2f} GB / {_ram_end.total/1e9:.2f} GB) -- "
      f"stayed under the {RAM_THRESHOLD_PCT}% ceiling: {_ram_end.percent < RAM_THRESHOLD_PCT}")
print(f"Total notebook wall-clock time: {_total_elapsed:.2f}s (real, measured -- no target speed asserted in advance).")
print(f"Notebook 05 complete. Results written to: {os.path.join(RESULTS_DIR, 'nb5_stress_test_report.json')}")
